# Task 4: feature engineering
Candidate feature blocks on top of the shipped `PairFeatures` (see `plan_t4.md`):
F1 calendar affinity, F2 time-local item similarity, F3 LightFM score, F4 combine + ablate.
Every block is φ(item1) − φ(item2), fitted on `tr` only, and scored with GB in both
directions of the validation halves.

In [1]:
%load_ext autoreload
%autoreload 2

import itertools
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from tqdm.auto import tqdm

from assignment_1.modules.models import ItemSimilarity, binary_cosine
from assignment_1.modules.models.ranking.lightFM import LightFMModel
from assignment_1.modules.tasks.task4 import task4_features
from assignment_1.modules.util import clip_timestamps, load_csv, make_pair_validation

In [2]:
DATA_DIR = "../data"  # notebook runs from notebooks/

train = load_csv("tasks_3_4_train.csv", DATA_DIR)
tr, pairs = make_pair_validation(train, include_timestamp=True, seed=0)
y = (pairs["which"] == pairs["item1"]).astype(int).to_numpy()
half = np.random.default_rng(0).random(len(pairs)) < 0.5
print(f"{len(tr):,} tr rows | {len(pairs):,} pairs")

1,286,318 tr rows | 142,924 pairs


## Evaluation
`evaluate` trains GB on one half and scores the other, in both directions. A→B matches
`task4.ipynb` (expect 0.9173 for the baseline).

In [3]:
def evaluate(X: np.ndarray, **gb) -> dict:
    out = {}
    for name, fit_on in [("A→B", half), ("B→A", ~half)]:
        m = HistGradientBoostingClassifier(random_state=0, **gb).fit(X[fit_on], y[fit_on])
        out[name] = round(float((m.predict(X[~fit_on]) == y[~fit_on]).mean()), 4)
    out["mean"] = round((out["A→B"] + out["B→A"]) / 2, 4)
    return out


def pair_diff(f) -> np.ndarray:
    """f(users, items, ts) -> (n, k) candidate features; returns f(item1) − f(item2)."""
    u, t = pairs["user"], pairs["timestamp"]
    return np.asarray(f(u, pairs["item1"], t) - f(u, pairs["item2"], t))


results = {}


def record(name: str, F: np.ndarray | None) -> dict:
    X = X0 if F is None else np.hstack([X0, F])
    r = evaluate(X)
    if name != "baseline":
        r["Δ"] = round(r["mean"] - results["baseline"]["mean"], 4)
    results[name] = r
    print(f"{name:<22} +{0 if F is None else F.shape[1]:>2} cols  {r}")
    return r

In [4]:
t0 = time.time()
X0 = task4_features().fit(tr).transform(pairs)
print(f"baseline features {X0.shape} in {time.time() - t0:.0f}s")
record("baseline", None)

baseline features (142924, 10) in 53s
baseline               + 0 cols  {'A→B': 0.9173, 'B→A': 0.9176, 'mean': 0.9174}


{'A→B': 0.9173, 'B→A': 0.9176, 'mean': 0.9174}

## F1. Calendar affinity
log[ P(part(t) | item) / P(part(t)) ], with P(· | item) smoothed toward the global
distribution (α pseudo-counts). Unknown items → 0.

In [5]:
def calendar_affinity(train: pd.DataFrame, part: str, alpha: float = 5.0):
    key = getattr(pd.to_datetime(clip_timestamps(train["timestamp"]), unit="s").dt, part)
    tbl = pd.crosstab(train["item"], key.to_numpy())
    glob = tbl.sum(0) / tbl.to_numpy().sum()
    P = np.log((tbl + alpha * glob).div(tbl.sum(1) + alpha, axis=0) / glob)

    def f(users, items, ts):
        k = getattr(pd.to_datetime(clip_timestamps(ts), unit="s").dt, part).to_numpy()
        r, c = P.index.get_indexer(items), P.columns.get_indexer(k)
        out = np.zeros(len(r))
        ok = (r >= 0) & (c >= 0)
        out[ok] = P.to_numpy()[r[ok], c[ok]]
        return out[:, None]

    return f

In [6]:
F_cal = {}
for part in ("weekday", "hour", "month"):
    F_cal[part] = pair_diff(calendar_affinity(tr, part))
    record(f"F1 {part}", F_cal[part])
record("F1 all calendar", np.hstack(list(F_cal.values())))

/tmp/ipykernel_6683/4124838371.py:2: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  key = getattr(pd.to_datetime(clip_timestamps(train["timestamp"]), unit="s").dt, part)
/tmp/ipykernel_6683/4124838371.py:4: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  glob = tbl.sum(0) / tbl.to_numpy().sum()
/tmp/ipykernel_6683/4124838371.py:5: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  P = np.log((tbl + alpha * glob).div(tbl.sum(1) + alpha, axis=0) / glob)
/tmp/ipykernel_6683/4124838371.py:8: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  k = getattr(pd.to_datetime(clip_timestamps(ts), unit="s").dt, part).to_numpy()
/tmp/ipykernel_6683/4124838371.py:8: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Seri

F1 weekday             + 1 cols  {'A→B': 0.917, 'B→A': 0.9176, 'mean': 0.9173, 'Δ': -0.0001}


/tmp/ipykernel_6683/4124838371.py:4: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  glob = tbl.sum(0) / tbl.to_numpy().sum()
/tmp/ipykernel_6683/4124838371.py:5: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  P = np.log((tbl + alpha * glob).div(tbl.sum(1) + alpha, axis=0) / glob)


F1 hour                + 1 cols  {'A→B': 0.917, 'B→A': 0.9172, 'mean': 0.9171, 'Δ': -0.0003}


/tmp/ipykernel_6683/4124838371.py:4: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  glob = tbl.sum(0) / tbl.to_numpy().sum()
/tmp/ipykernel_6683/4124838371.py:5: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  P = np.log((tbl + alpha * glob).div(tbl.sum(1) + alpha, axis=0) / glob)


F1 month               + 1 cols  {'A→B': 0.917, 'B→A': 0.9177, 'mean': 0.9173, 'Δ': -0.0001}
F1 all calendar        + 3 cols  {'A→B': 0.9168, 'B→A': 0.9173, 'mean': 0.917, 'Δ': -0.0004}


{'A→B': 0.9168, 'B→A': 0.9173, 'mean': 0.917, 'Δ': -0.0004}

## F2. Time-local item similarity
For each (user, t): the user's `n_side` history interactions just before t and `n_side`
just after. Cosine similarity (fitted on `tr`) of the candidate to those items, summarized
by time window, by nearest-k, and with exponential decay in |Δt|.

In [7]:
class LocalSimilarity:
    def __init__(self, similarity=binary_cosine, n_side: int = 10,
                 windows=(1, 7, 30), k_near=(1, 3, 10), tau: float = 7.0):
        self.similarity, self.n_side = similarity, n_side
        self.windows, self.k_near, self.tau = windows, k_near, tau

    def fit(self, train: pd.DataFrame):
        self.sim = ItemSimilarity(self.similarity).fit(train)
        self.S = self.sim.S.tocsr()
        u = self.sim.users.get_indexer(train["user"])
        i = self.sim.items.get_indexer(train["item"])
        t = clip_timestamps(train["timestamp"]).to_numpy(float)
        o = np.lexsort((t, u))
        self.u_s, self.i_s, self.t_s = u[o], i[o], t[o]
        assert len(self.sim.users) < 9e5  # key = user * 1e10 + t stays exact in float64
        self.key = self.u_s * 1e10 + self.t_s
        self.start = np.searchsorted(self.u_s, np.arange(len(self.sim.users) + 1))
        return self

    def __call__(self, users, items, ts) -> pd.DataFrame:
        u = self.sim.users.get_indexer(users)
        i = self.sim.items.get_indexer(items)
        t = clip_timestamps(ts).to_numpy(float)
        known = (u >= 0) & (i >= 0)
        u0, i0 = np.where(known, u, 0), np.where(known, i, 0)

        # 2·n_side neighbours in time from the user's sorted history
        p = np.searchsorted(self.key, u0 * 1e10 + t)
        idx = p[:, None] + np.arange(-self.n_side, self.n_side)
        valid = ((idx >= self.start[u0][:, None]) & (idx < self.start[u0 + 1][:, None])
                 & known[:, None])
        idx = np.clip(idx, 0, len(self.i_s) - 1)
        dt = np.where(valid, np.abs(self.t_s[idx] - t[:, None]) / 86400, np.inf)
        h = self.i_s[idx]
        s = np.asarray(self.S[np.repeat(i0, idx.shape[1]), h.ravel()]).reshape(idx.shape)
        s = np.where(valid, s, 0.0)

        out = {}
        for w in self.windows:
            m = dt <= w
            out[f"loc_w{w}_sum"] = (s * m).sum(1)
            out[f"loc_w{w}_max"] = (s * m).max(1)
        near = np.argsort(dt, axis=1, kind="stable")
        s_near = np.take_along_axis(s, near, 1)
        n_valid = valid.sum(1)
        for k in self.k_near:
            out[f"loc_k{k}_mean"] = s_near[:, :k].sum(1) / np.clip(np.minimum(n_valid, k), 1, None)
            out[f"loc_k{k}_max"] = s_near[:, :k].max(1)
        out["loc_decay"] = (s * np.exp(-dt / self.tau)).sum(1)
        return pd.DataFrame(out)

In [8]:
t0 = time.time()
local = LocalSimilarity().fit(tr)
F_loc = pair_diff(local)
print(f"local similarity in {time.time() - t0:.0f}s")
record("F2 local sim", F_loc)

local similarity in 23s
F2 local sim           +13 cols  {'A→B': 0.9218, 'B→A': 0.9237, 'mean': 0.9227, 'Δ': 0.0053}


{'A→B': 0.9218, 'B→A': 0.9237, 'mean': 0.9227, 'Δ': 0.0053}

Sensitivity: neighbourhood size and decay.

In [9]:
for n_side, tau in [(5, 7.0), (20, 7.0), (10, 1.0), (10, 30.0)]:
    F = pair_diff(LocalSimilarity(n_side=n_side, tau=tau).fit(tr))
    record(f"F2 n={n_side} τ={tau}", F)

F2 n=5 τ=7.0           +13 cols  {'A→B': 0.9218, 'B→A': 0.9235, 'mean': 0.9226, 'Δ': 0.0052}
F2 n=20 τ=7.0          +13 cols  {'A→B': 0.9219, 'B→A': 0.9237, 'mean': 0.9228, 'Δ': 0.0054}
F2 n=10 τ=1.0          +13 cols  {'A→B': 0.9225, 'B→A': 0.9233, 'mean': 0.9229, 'Δ': 0.0055}
F2 n=10 τ=30.0         +13 cols  {'A→B': 0.9225, 'B→A': 0.9234, 'mean': 0.9229, 'Δ': 0.0055}


## F3. LightFM score
Side-feature builders return {id: [feature names]}; LightFM keeps the identity features
and adds these on top.

In [10]:
def item_side(train: pd.DataFrame) -> dict:
    pop = train["item"].value_counts()
    ts = pd.to_datetime(clip_timestamps(train["timestamp"]), unit="s")
    year = ts.groupby(train["item"]).min().dt.year
    dow = pd.crosstab(train["item"], ts.dt.weekday.to_numpy()).idxmax(1)
    pb = np.floor(np.log2(pop)).astype(int)
    return {i: [f"pop:{pb[i]}", f"year:{year[i]}", f"dow:{dow[i]}"] for i in pop.index}


def user_side(train: pd.DataFrame) -> dict:
    n = train["user"].value_counts()
    ts = pd.to_datetime(clip_timestamps(train["timestamp"]), unit="s")
    year = ts.dt.year.groupby(train["user"]).median().round().astype(int)
    nb = np.floor(np.log2(n)).astype(int)
    return {u: [f"act:{nb[u]}", f"uyear:{year[u]}"] for u in n.index}


def lightfm_feature(**params):
    m = LightFMModel(**params).fit(tr)
    return lambda u, i, t: np.nan_to_num(m(u.to_numpy(), i.to_numpy()))[:, None]

In [11]:
variants = {
    "bpr": {},
    "bpr + item": {"item_features": item_side},
    "bpr + item + user": {"item_features": item_side, "user_features": user_side},
    "warp": {"loss": "warp"},
    "warp + item + user": {"loss": "warp", "item_features": item_side,
                           "user_features": user_side},
}
F_lfm = {}
for name, kw in tqdm(variants.items()):
    t0 = time.time()
    F_lfm[name] = pair_diff(lightfm_feature(no_components=64, epochs=30, **kw))
    print(f"  fit+score {time.time() - t0:.0f}s")
    record(f"F3 {name}", F_lfm[name])

  0%|          | 0/5 [00:00<?, ?it/s]

  fit+score 15s
F3 bpr                 + 1 cols  {'A→B': 0.917, 'B→A': 0.9177, 'mean': 0.9173, 'Δ': -0.0001}


/tmp/ipykernel_6683/3255909997.py:5: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  dow = pd.crosstab(train["item"], ts.dt.weekday.to_numpy()).idxmax(1)


  fit+score 61s
F3 bpr + item          + 1 cols  {'A→B': 0.9172, 'B→A': 0.9179, 'mean': 0.9176, 'Δ': 0.0002}


/tmp/ipykernel_6683/3255909997.py:5: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  dow = pd.crosstab(train["item"], ts.dt.weekday.to_numpy()).idxmax(1)


  fit+score 69s
F3 bpr + item + user   + 1 cols  {'A→B': 0.9172, 'B→A': 0.918, 'mean': 0.9176, 'Δ': 0.0002}
  fit+score 13s
F3 warp                + 1 cols  {'A→B': 0.918, 'B→A': 0.9186, 'mean': 0.9183, 'Δ': 0.0009}


/tmp/ipykernel_6683/3255909997.py:5: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  dow = pd.crosstab(train["item"], ts.dt.weekday.to_numpy()).idxmax(1)


  fit+score 44s
F3 warp + item + user  + 1 cols  {'A→B': 0.9171, 'B→A': 0.9173, 'mean': 0.9172, 'Δ': -0.0002}


Grid for the best variant (set `BEST_LFM` from the table above).

In [12]:
BEST_LFM = "bpr + item"
for k, ep in tqdm(list(itertools.product((32, 64), (20, 50)))):
    F = pair_diff(lightfm_feature(no_components=k, epochs=ep, **variants[BEST_LFM]))
    F_lfm[f"{BEST_LFM} k={k} ep={ep}"] = F
    record(f"F3 {BEST_LFM} k={k} ep={ep}", F)

  0%|          | 0/4 [00:00<?, ?it/s]

/tmp/ipykernel_6683/3255909997.py:5: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  dow = pd.crosstab(train["item"], ts.dt.weekday.to_numpy()).idxmax(1)


F3 bpr + item k=32 ep=20 + 1 cols  {'A→B': 0.9176, 'B→A': 0.9183, 'mean': 0.918, 'Δ': 0.0006}


/tmp/ipykernel_6683/3255909997.py:5: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  dow = pd.crosstab(train["item"], ts.dt.weekday.to_numpy()).idxmax(1)


F3 bpr + item k=32 ep=50 + 1 cols  {'A→B': 0.9171, 'B→A': 0.9177, 'mean': 0.9174, 'Δ': 0.0}


/tmp/ipykernel_6683/3255909997.py:5: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  dow = pd.crosstab(train["item"], ts.dt.weekday.to_numpy()).idxmax(1)


F3 bpr + item k=64 ep=20 + 1 cols  {'A→B': 0.9173, 'B→A': 0.9178, 'mean': 0.9175, 'Δ': 0.0001}


/tmp/ipykernel_6683/3255909997.py:5: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  dow = pd.crosstab(train["item"], ts.dt.weekday.to_numpy()).idxmax(1)


F3 bpr + item k=64 ep=50 + 1 cols  {'A→B': 0.9174, 'B→A': 0.9172, 'mean': 0.9173, 'Δ': -0.0001}


## F4. Combine and ablate
Set `KEEP` to the blocks that gained ≥ +0.002 in both directions.

In [13]:
blocks = {
    "calendar": np.hstack(list(F_cal.values())),
    "local": F_loc,
    "lightfm": F_lfm[BEST_LFM],
}
KEEP = list(blocks)  # edit after reading the results above

X_all = np.hstack([X0] + [blocks[b] for b in KEEP])
record("F4 all kept", X_all[:, X0.shape[1]:])
for b in KEEP:
    rest = [blocks[o] for o in KEEP if o != b]
    record(f"F4 − {b}", np.hstack(rest) if rest else None)

F4 all kept            +17 cols  {'A→B': 0.9225, 'B→A': 0.9233, 'mean': 0.9229, 'Δ': 0.0055}
F4 − calendar          +14 cols  {'A→B': 0.9224, 'B→A': 0.9236, 'mean': 0.923, 'Δ': 0.0056}
F4 − local             + 4 cols  {'A→B': 0.9172, 'B→A': 0.9175, 'mean': 0.9173, 'Δ': -0.0001}
F4 − lightfm           +16 cols  {'A→B': 0.9218, 'B→A': 0.9232, 'mean': 0.9225, 'Δ': 0.0051}


GB tuning on the combined set (more features may want more trees).

In [14]:
rows = []
for it, lr, leaves in tqdm(list(itertools.product((100, 300), (0.05, 0.1), (31, 63)))):
    r = evaluate(X_all, max_iter=it, learning_rate=lr, max_leaf_nodes=leaves)
    rows.append({"max_iter": it, "learning_rate": lr, "max_leaf_nodes": leaves, **r})
pd.DataFrame(rows).sort_values("mean", ascending=False)

  0%|          | 0/8 [00:00<?, ?it/s]

,max_iter,learning_rate,max_leaf_nodes,A→B,B→A,mean
5,300,0.05,63,0.9227,0.9244,0.9235
2,100,0.10,31,0.9225,0.9233,0.9229
6,300,0.10,31,0.9226,0.9233,0.9229
1,100,0.05,63,0.9218,0.9238,0.9228
4,300,0.05,31,0.9220,0.9236,0.9228
3,100,0.10,63,0.9219,0.9232,0.9225
7,300,0.10,63,0.9219,0.9232,0.9225
0,100,0.05,31,0.9211,0.9229,0.9220


In [15]:
pd.DataFrame(results).T

,A→B,B→A,mean,Δ
baseline,0.9173,0.9176,0.9174,NaN
F1 weekday,0.9170,0.9176,0.9173,-0.0001
F1 hour,0.9170,0.9172,0.9171,-0.0003
F1 month,0.9170,0.9177,0.9173,-0.0001
F1 all calendar,0.9168,0.9173,0.9170,-0.0004
F2 local sim,0.9218,0.9237,0.9227,0.0053
F2 n=5 τ=7.0,0.9218,0.9235,0.9226,0.0052
F2 n=20 τ=7.0,0.9219,0.9237,0.9228,0.0054
F2 n=10 τ=1.0,0.9225,0.9233,0.9229,0.0055
F2 n=10 τ=30.0,0.9225,0.9234,0.9229,0.0055
